In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 1/12 - imports & load
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# sklearn
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import mean_squared_error

# stats
from scipy.stats import skew

# xgboost
import xgboost as xgb

# reproducibility
RSEED = 42
np.random.seed(RSEED)

# paths (Kaggle default location)
INPUT_DIR = '/kaggle/input/house-prices-advanced-regression-techniques'
train = pd.read_csv(f'{INPUT_DIR}/train.csv')
test  = pd.read_csv(f'{INPUT_DIR}/test.csv')

print("train shape:", train.shape, " test shape:", test.shape)


In [ ]:
# 2/12 - target and ids
train_id = train['Id']
test_id  = test['Id']

y = train['SalePrice'].copy()
print("Target: min", y.min(), "max", y.max(), "median", y.median())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# SalePrice distribution
plt.figure(figsize=(8,4))
sns.histplot(y, kde=True, bins=40)
plt.title("Distribution of SalePrice")
plt.show()

# Log-transformed target
plt.figure(figsize=(8,4))
sns.histplot(np.log1p(y), kde=True, bins=40)
plt.title("Distribution of log(SalePrice)")
plt.show()


In [ ]:
# 3/12 - combine (drop Id and SalePrice from train)
X_all = pd.concat([
    train.drop(['Id','SalePrice'], axis=1),
    test.drop(['Id'], axis=1)
], axis=0, ignore_index=True)

# Example engineered features (safe, helpful)
X_all['TotalSF'] = X_all['TotalBsmtSF'].fillna(0) + X_all['1stFlrSF'] + X_all['2ndFlrSF']
X_all['TotalBath'] = X_all[['FullBath','HalfBath','BsmtFullBath','BsmtHalfBath']].fillna(0).eval(
    "FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath"
)
X_all['HasPool'] = (X_all['PoolArea'].fillna(0) > 0).astype(int)
X_all['HasGarage'] = (X_all['GarageArea'].fillna(0) > 0).astype(int)
X_all['HouseAge'] = X_all['YrSold'] - X_all['YearBuilt']
X_all['RemodAge']  = X_all['YrSold'] - X_all['YearRemodAdd']
X_all['TotalPorch'] = X_all[['OpenPorchSF','EnclosedPorch','3SsnPorch','ScreenPorch']].fillna(0).sum(axis=1)

print("Combined shape after FE:", X_all.shape)


In [ ]:
# % missing values
missing = train.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(8,6))
sns.barplot(x=missing.values, y=missing.index)
plt.title("Missing Values in Train Data (Proportion)")
plt.xlabel("Fraction Missing")
plt.show()


In [ ]:
# 4/12 - fill categorical 'None' for many features where NA means "no X"
fill_none = [
 'Alley','PoolQC','Fence','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond',
 'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType'
]
for c in fill_none:
    if c in X_all.columns:
        X_all[c] = X_all[c].fillna('None')

# MasVnrArea -> zero if NA (no veneer)
if 'MasVnrArea' in X_all.columns:
    X_all['MasVnrArea'] = X_all['MasVnrArea'].fillna(0)

# LotFrontage -> impute by neighborhood median
if 'LotFrontage' in X_all.columns and 'Neighborhood' in X_all.columns:
    X_all['LotFrontage'] = X_all.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )

# GarageYrBlt -> median (safe)
if 'GarageYrBlt' in X_all.columns:
    X_all['GarageYrBlt'] = X_all['GarageYrBlt'].fillna(X_all['GarageYrBlt'].median())

# Fill any remaining object columns with mode (safe fallback)
obj_cols = X_all.select_dtypes(include=['object']).columns
for c in obj_cols:
    X_all[c] = X_all[c].fillna(X_all[c].mode().iloc[0])


In [ ]:
# 5/12 - detect skew and log1p transform high-skew numeric features
numeric_feats = X_all.select_dtypes(include=[np.number]).columns.tolist()

skewed = X_all[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewed_feats = skewed[abs(skewed) > 0.75].index.tolist()
print("Applying log1p to", len(skewed_feats), "skewed features")

for feat in skewed_feats:
    # log1p is safe for zeros; ensure no negative values
    if (X_all[feat] >= 0).all():
        X_all[feat] = np.log1p(X_all[feat])
    else:
        # skip if negative values would break log1p
        pass


In [ ]:
# 6/12 - column lists for transformer
numeric_feats = X_all.select_dtypes(include=[np.number]).columns.tolist()
categorical_feats = X_all.select_dtypes(include=['object']).columns.tolist()

# handle different sklearn versions for OneHotEncoder param name
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    # older sklearn
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

from sklearn.preprocessing import FunctionTransformer
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('ohe', ohe)
])


In [ ]:
# 7/12 - build preprocessor and transform combined data
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numeric_feats),
    ('cat', cat_transformer, categorical_feats)
], remainder='drop')

X_all_trans = preprocessor.fit_transform(X_all)   # fit on combined to avoid unseen categories

n_train = train.shape[0]
X_train = X_all_trans[:n_train]
X_test  = X_all_trans[n_train:]

print("X_train shape:", X_train.shape, "X_test shape:", X_test.shape)


In [ ]:
# 8/12 - log-transform target and CV helper
y_log = np.log1p(y)

from sklearn.model_selection import KFold
def rmse_cv(model, X, y, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RSEED)
    scores = -cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=kf, n_jobs=-1)
    return scores


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Models
lasso = Lasso(alpha=0.001, max_iter=5000, random_state=RSEED)
rf    = RandomForestRegressor(n_estimators=200, random_state=RSEED, n_jobs=-1)
gbr   = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=3, random_state=RSEED)
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=RSEED,
    n_jobs=-1,
    eval_metric='rmse'
)

# RMSE CV for each
models = {'Lasso': lasso, 'RandomForest': rf, 'GradientBoosting': gbr, 'XGBoost': xgb_model}
rmse_results = {}

for name, model in models.items():
    scores = rmse_cv(model, X_train, y_log)
    rmse_results[name] = (scores.mean(), scores.std())
    print(f"{name}: mean RMSE={scores.mean():.5f}, std={scores.std():.5f}")

# Table
rmse_table = pd.DataFrame(rmse_results, index=['mean','std']).T.sort_values('mean')
print("\nModel Comparison (log RMSE):")
display(rmse_table)

# Fit best model(s) on full train
best_model = xgb_model.fit(X_train, y_log)


In [ ]:
# Feature importance plot
xgb_importance = best_model.feature_importances_
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_importance
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(8,5))
sns.barplot(x='importance', y='feature', data=importance_df)
plt.title("Top 10 Feature Importances (XGBoost)")
plt.show()


In [ ]:
# Train final Lasso model on full training data
lasso_final = LassoCV(
    alphas=np.logspace(-4, -0.5, 30),
    cv=5,
    random_state=42
).fit(X_train, y_log)

# Train final XGB model on full training data
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse'
).fit(X_train, y_log)


In [ ]:
# 11/12 - predict on test and average
preds_lasso = lasso_final.predict(X_test)
preds_xgb   = xgb_model.predict(X_test)

preds_log_mean = (preds_lasso + preds_xgb) / 2.0
final_preds = np.expm1(preds_log_mean)   # back to dollars

# quick sanity
print("Preds: min", final_preds.min(), "max", final_preds.max(), "mean", final_preds.mean())


In [ ]:
# 12/12 - write submission
submission = pd.DataFrame({'Id': test_id, 'SalePrice': final_preds})
submission.to_csv('submission.csv', index=False)
print("Wrote submission.csv (rows):", submission.shape[0])


In [ ]:
# Ensemble: average Lasso, RF, and XGB predictions
lasso.fit(X_train, y_log)
rf.fit(X_train, y_log)

preds_lasso = lasso.predict(X_test)
preds_rf    = rf.predict(X_test)
preds_xgb   = best_model.predict(X_test)

preds_log_mean = (preds_lasso + preds_rf + preds_xgb) / 3.0
final_preds = np.expm1(preds_log_mean)

submission = pd.DataFrame({'Id': test_id, 'SalePrice': final_preds})
submission.to_csv('submission.csv', index=False)
print("Submission file written:", submission.shape)
